In [ ]:
# Setup — all imports live here so the notebook executes top-to-bottom
# without reimporting mid-notebook. When running headlessly (CI, HPC),
# uncomment the ``matplotlib.use('Agg')`` line *before* the pyplot import
# so figures never need an interactive display.
import dataclasses
import logging
import os
import sys
from pathlib import Path

# import matplotlib
# matplotlib.use('Agg')  # enable for headless runs
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import seaborn as sns

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from IPython.display import display  # noqa: E402
from scipy.stats import wilcoxon  # noqa: E402

from scripts.notebook_helpers import (  # noqa: E402
    WAVELET_FREQ_MAX,
    WAVELET_FREQ_MIN,
    WAVELET_N_FREQS,
    load_joined_condition_wavelets,
    load_paired_condition_wavelets,
    resolve_notebook_wavelet_cache_dir,
    resolve_wavelet_dir,
)
from src.analysis import iva_quality  # noqa: E402
from src.analysis.assr_trials import reference_snr_tests  # noqa: E402
from src.analysis.wavelet_ica import zscore_by_time  # noqa: E402
from src.definitions.constants import AssrEpoch, ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    REAL_CONDITIONS,
    ConditionVariants,
    CoordinateSystems,
    ExclusionCategories,
    ExperimentNames,
    MusicTypeVariants,
    PreprocessedDataVariants,
)
from src.io.loading import assr_electrode_mask  # noqa: E402
from src.visualization.iva_quality_plots import (  # noqa: E402
    participant_sort_key,
    save_fig,
    topo_info_subset,
)
from src.analysis.wavelet_jica import (  # noqa: E402
    assemble_join,
    fit_joint_ica,
    SIGNAL_VARIANTS,
    VARIANT_RECIPE,
    VARIANT_REFERENCE_FROM,
    VARIANT_UNITS,
)
from src.visualization.jica_plots import (  # noqa: E402
    plot_pvalue_summary,
    plot_response_courses,
    plot_snr_vs_reference,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
mne.set_log_level("ERROR")
%matplotlib inline
print("Setup complete.")

# Analysis of the Joint-ICA Components — 40 Hz Response, Both Conditions

The ICA counterpart of
[`iva_component_analysis_joined_tracks.ipynb`](../06-iva-condition-comparison/iva_component_analysis_joined_tracks.ipynb)
and its sibling: it takes the components the jICA decomposition notebooks recover and asks
the question the whole stage exists for — **is the 40 Hz steady-state response lower under
Psilocybin than under Placebo, and does a learned component see that better than simply
averaging the ASSR electrodes?**

The two products are the ones the IVA stage produces:

1. **The response over time, across stimuli** — one panel per spatial filter, Placebo and
   Psilocybin overlaid, mean ± SEM **across participants**, with the driven interval
   shaded. This is the figure to read first.
2. **The participant-level tests** — exact Wilcoxon signed-rank, paired over participants:
   `Placebo − Psilocybin` per source (one-sided), each component's condition contrast
   against the fixed-electrode reference (two-sided), and each component's SNR against that
   reference within each condition (one-sided).

## Why this notebook decomposes rather than loads

The IVA analysis notebooks read a *stored* decomposition, written on a node by
`scripts/run_iva_condition_tracks.py --store_components`. There is no ICA component store,
and adding one would buy nothing here: the expensive part is reading the wavelet caches,
which this notebook has to do anyway — the trials are cut from the **raw, un-z-scored**
wavelet power, not from anything a store would hold. So Step 1 runs the same jICA as
[`wavelet_ica_channel_joined.ipynb`](wavelet_ica_channel_joined.ipynb) /
[`wavelet_ica_channel_joined_tracks.ipynb`](wavelet_ica_channel_joined_tracks.ipynb), and
everything downstream is new.

`JOIN` selects which of the two joins to analyse, and the rest of the notebook is identical
either way — because both end at the same place: **one spatial filter per (participant,
condition)**.

| | `JOINED_TRACKS` (default) | `JOINED` |
|---|---|---|
| Channel blocks | one per participant | one per **recording** (participant × condition) |
| Filter used for a condition | the participant's, **shared** | that recording's own |
| A condition difference can come from | the signal only | the signal *or* the filter |
| Polarity anchor | one per (participant, component) | one per (participant, condition, component) |

`JOINED_TRACKS` is the default because it is the cleaner contrast: the same operator reads
both conditions, so a difference cannot be an artefact of the filter having changed. Run
`JOINED` as the check that the conclusion survives when each recording gets its own filter.

## What the per-recording filter is, exactly

This is the one place jICA differs structurally from IVA, and it has to be said plainly
before any number is read. FastICA's unmixing matrix acts on the **stacked** channel
vector, so for component *k*

```
source_k(f, t)  =  Σ_s  u_{s,k} · x_s(f, t)
                   ^^^^^^^^^^^^^^^^^^^^^^^
                   summed over every recording s
```

`u_{s,k}` — row *k* of the unmixing matrix restricted to recording *s*'s channel block — is
therefore recording *s*'s **contribution** to the shared source, not "that recording's own
copy of component *k*" (which is what IVA gives). What this notebook projects is exactly
that contribution: a `(C,)` weighting over that recording's channels, applied to that
recording's own wavelet power. It is a well-defined per-recording signal, it is the only
per-recording read-out jICA offers, and it is what the loadings in the decomposition
notebooks measure the size of — but it is a partial term of a group quantity, so a
component with a near-zero loading on a participant has a near-noise time course for them.

## The reference, and why there is only one

The fixed fronto-central electrode average from
[`config/assr_electrodes/`](../../config/assr_electrodes) — the selection the 40 Hz
steady-state response is conventionally read from. It is the same *kind* of operator as a
learned filter (a weighting over channels that contracts the channel axis and leaves
frequency and time untouched) but fixed by the montage and the paradigm, so it needs no
estimation and carries no sign ambiguity. Every component is judged against it.

The IVA notebooks carry a *second* reference, the same mask projected onto each
participant's own channel-PCA subspace, so that the mask can only see what the reduction
kept. **That has no clean analogue here.** jICA's whitening truncation retains a subspace of
the *joint* `S·C`-dimensional channel space, so projecting a per-recording electrode
average into it would mix in other participants' channels — a reference nobody would want.
The fixed mask on the real channel axis is therefore the single reference, and the
comparisons below are against it.

## What is done to the signal, and what is not

Three normalisations are carried side by side (`TEST_VARIANTS`). They differ in two
independent ways — which signal the filter is applied to, and whether each trial is
referenced to its own pre-stimulus window — and the three of them fill the reachable cells
of that 2×2:

| | no per-trial baseline | per-trial pre-stimulus baseline |
|---|---|---|
| raw wavelet power | — | **`prestim`** |
| z-scored (what the fit saw) | **`zscored`** | **`zscored_prestim`** |

- **`prestim`** — the **raw** wavelet power is projected, cut into trials, and every trial
  is referenced to its **own** pre-stimulus interval: `(x − baseline) / baseline SD`. Units
  of pre-stimulus SD. Its one flaw: applying the filter to *un*-z-scored power drops the
  per-channel `1/sd` weighting the fit folded in, so this **approximates** the component.
- **`zscored`** — the filter is applied to exactly the signal the decomposition saw. That
  makes it the component *itself*, not an approximation: the per-block terms
  `u_{b,k} · x_b^z` sum over blocks to the global TF map. Already dimensionless, so no
  per-trial baseline.
- **`zscored_prestim`** — that exact component, then referenced to each trial's own
  pre-stimulus window, which puts it in the same pre-stimulus-SD units as the fixed
  reference so the two can be read on one axis. The counterpart of the 06 IVA notebook's
  `stored_prestim`.

**The fixed reference row is held still.** In `zscored_prestim` the `ASSR-mask` row is
`prestim`'s verbatim, so the two variants are judged against exactly the same numbers and
the comparison isolates one thing: how the **component** was derived, and nothing else.
Requesting `zscored_prestim` therefore requires `prestim`.

**Why there is no test on the global TF maps themselves**, which is what the IVA stage's
`stored_prestim` reads. jICA's `tf_maps` is `(K, F, T)` — one map per component, *shared*
by every recording — so cutting trials from it yields one number per (component, condition)
with **no participant axis**. A paired test over participants on that would report `n = P`
for what is a single measurement. The per-participant quantity jICA does offer is the
per-block term of the sum that *forms* those maps, which is exactly what `zscored` and
`zscored_prestim` test. And the trial-count gain the IVA variant buys (~148 vs ~9) comes
here from running the full time axis instead, not from a different route to the source.

Every variant is computed on the same trials and the same participants, so the caveat the
IVA stage carries — component rows averaging ~148 trials against mask rows averaging ~9 —
does not apply to any of them.

## Variables produced

| Variable | Shape | Description |
|----------|-------|-------------|
| `raw_by_condition` | `(P, C, F, T)` each | Un-z-scored wavelet power, participant-matched |
| `filters` | `(K, B, C)` | Per-block spatial filters (the unmixing rows, split by block) |
| `patterns` | `(B, K, C)` | Per-block forward patterns — topographies, and the polarity anchor |
| `assr_filter` | `(C,)` | The fixed ASSR-electrode weighting |
| `trials[variant][sel][cond]` | `(P, S, N, W)` | Stimulus-locked 40 Hz trials, `S = K + 1` sources |
| `value_by_variant[variant][cond]` | `(P, S)` | One response per (participant, source) — what every test reduces to |
| `course_by_variant[variant][cond]` | `(P, S, W)` | Per-participant median epoch course — what Figure 1 draws |

## Configuration

In [ ]:
# ── Which join to analyse ──────────────────────────────────────
# ConditionVariants.JOINED_TRACKS — one channel block per participant, so the SAME filter
#   reads both conditions and a difference cannot come from the filter.
# ConditionVariants.JOINED       — one block per recording, so each condition is read by
#   its own filter. Run it as the robustness check.
JOIN = ConditionVariants.JOINED_TRACKS

# ── Experiment configuration ───────────────────────────────────
EXPERIMENT_NAME = ExperimentNames.ASSR
CONDITIONS_TO_POOL = list(REAL_CONDITIONS)  # Placebo first, then Psilocybin
if EXPERIMENT_NAME == ExperimentNames.ASSR:
    MUSIC_TYPE = MusicTypeVariants.ASSR
else:
    MUSIC_TYPE = MusicTypeVariants.CLASSICAL
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]
# Needed to resolve the ASSR-electrode include-list against the channel names.
COORDINATE_SYSTEM = CoordinateSystems.HYDROGEL_257_NO_FIDUCIALS

# ── Wavelet settings ─────────────────────────────────────────
REPRESENTATION = "power"
FREQS = np.linspace(WAVELET_FREQ_MIN, WAVELET_FREQ_MAX, WAVELET_N_FREQS)
REUSE_WAVELETS = True

# ── Cohort, channel and time extent ────────────────────────────
# These are NOT free knobs here, and the defaults are the working ones:
#
# * N_CHANNELS_SUBSET = None (every channel). The ASSR-electrode reference is a
#   fronto-central selection spread over the montage, so a leading channel slice does not
#   contain it — with a 32-channel subset the mask selects almost nothing and the
#   comparison the whole notebook is built on becomes meaningless. Step 3 refuses rather
#   than quietly averaging over two electrodes.
# * N_PAIRS_SUBSET = None (every participant). The participant is the unit of every test,
#   and an exact Wilcoxon over P participants cannot return a p below 2/2**P: 0.0005 at
#   P = 12, but 0.06 at P = 5. Subsetting the cohort does not just weaken the test, it
#   makes it unable to reject at all.
# * N_TIMES_SUBSET is the real constraint, and it is a trade: it bounds how many stimulus
#   onsets fit the cached window (~9 trials per condition at 3000 samples = 12 s), while
#   the memory below scales linearly in it.
#
# Peak memory ~ 3 x (P_blocks x C x F x T) x itemsize: the raw tensor, the z-scored copy
# and the FastICA input. At 12 participants, 195 channels, 50 frequencies, 3000 samples
# and float32 that is ~8 GB, so this notebook belongs on a node. CAST_FLOAT32 halves it
# if the cache happens to be float64.
N_PAIRS_SUBSET: int | None = None
N_CHANNELS_SUBSET: int | None = None
N_TIMES_SUBSET: int | None = 3000
CAST_FLOAT32 = True

# ── ICA settings ──────────────────────────────────────────────
# Fewer components than the decomposition notebooks' 10, deliberately: every component
# adds an uncorrected test to two families below, and FastICA converges more reliably at
# small K. Step 1 reports convergence; a non-converged unmixing is wherever the solver
# stopped, not a fixed point, and nothing downstream is worth reading from it.
N_ICA = 5
ICA_ALGORITHM = "parallel"  # or "deflation"
ICA_FUN = "logcosh"
ICA_MAX_ITER = 2000
ICA_TOL = 1e-4
RANDOM_STATE = 42

# ── The fixed ASSR-electrode reference ────────────────────────
ASSR_MASK_LABEL = "ASSR-mask"
# True averages the selected electrodes, False sums them. The mean keeps the output in the
# input's power units and does not scale with how many electrodes the list happens to
# contain, which a raw binary sum does.
ASSR_MASK_NORMALIZE = True
# Preprocessing drops the boundary electrodes, so a listed electrode can be absent from a
# recording. True refuses to under-select; False accepts the intersection and logs what was
# missing. Start at True and only relax it once the log says which electrodes are gone.
ASSR_MASK_STRICT = True

# ── The frequency selections the response is read at ──────────
# (label, centre Hz, half-width Hz). A zero half-width takes the single nearest bin — on
# this 1 Hz grid that IS the 40 Hz row. A positive one averages every bin inside the closed
# interval, which is what survives wavelet smearing and a few Hz of stimulator drift. Both
# are extracted because they answer the same question with different exposure to that
# smearing, and they cost almost nothing.
FREQ_SELECTIONS: list[tuple[str, float, float]] = [
    ("40hz", iva_quality.ASSR_FREQ, 0.0),
    (
        f"{iva_quality.ASSR_FREQ - iva_quality.TF_ANCHOR_HALFWIDTH_HZ:g}"
        f"-{iva_quality.ASSR_FREQ + iva_quality.TF_ANCHOR_HALFWIDTH_HZ:g}hz",
        iva_quality.ASSR_FREQ,
        iva_quality.TF_ANCHOR_HALFWIDTH_HZ,
    ),
]

# ── Where the stimulus onsets come from ───────────────────────
# The sidecar of the concatenated array — the array the wavelet transform was run on — so
# its sample indices address exactly the time axis the projected tracks are on. No
# shifting, no rescaling.
CONCATENATED_DIR: Path = (
    ProjectPaths.PROCESSED_DATA_DIR
    / EXPERIMENT_NAME.value
    / PreprocessedDataVariants.CONCATENATED.value
)
MIN_TRIALS_EXPECTED = 5

# ── What the tests reduce over ────────────────────────────────
# Which frequency selection: any label in FREQ_SELECTIONS.
TEST_SELECTION = "40hz"
# The stimulus window, in seconds. None = the paradigm's full driven interval
# (0 - AssrEpoch.STIMULUS_DURATION_S). A (start, stop) tuple overrides it, e.g. (0.2, 0.5)
# to skip the onset transient and read only the sustained steady state.
TEST_STIMULUS_INTERVAL: tuple[float, float] | None = None
# How a trial's time course becomes one number.
#   "stimulus"            mean over the driven interval.
#   "stimulus_minus_rest" that minus the mean over the rest of the epoch, which cancels
#                         whatever offset the per-trial normalisation left behind.
# Fix this BEFORE looking at any p-value.
RESPONSE_MEASURE = "stimulus"
# The signal-normalisation variants, run side by side. They differ in two independent
# ways — which signal the filter is applied to, and whether each trial is referenced to
# its own pre-stimulus window — and the three of them fill the reachable cells of that
# 2x2 (see src.analysis.wavelet_jica.SIGNAL_VARIANTS):
#
#                        | no per-trial baseline | per-trial pre-stimulus baseline
#     -------------------+-----------------------+--------------------------------
#     raw wavelet power  |          --           | "prestim"
#     z-scored (the fit) | "zscored"             | "zscored_prestim"
#
#   "prestim"         the RAW power is projected and every trial referenced to its own
#                     pre-stimulus SD (an SNR). Its flaw: applying the filter to
#                     un-z-scored power drops the per-channel 1/sd weighting the fit
#                     folded in, so it APPROXIMATES the component.
#   "zscored"         the filter is applied to exactly the signal the fit saw, so the
#                     per-block terms u_{b,k}.x_b^z ARE the component — they sum over
#                     blocks to the global TF map. No per-trial baseline needed.
#   "zscored_prestim" that exact component, then put into pre-stimulus-SD units so it
#                     shares an axis with the fixed reference. The counterpart of the
#                     06 IVA notebook's "stored_prestim".
#
# THE FIXED REFERENCE ROW IS HELD STILL. In "zscored_prestim" the ASSR-mask row is
# "prestim"'s verbatim, so the two variants are compared against exactly the same
# numbers and the comparison isolates one thing: how the COMPONENT was derived.
# Requesting "zscored_prestim" therefore requires "prestim".
TEST_VARIANTS = ["prestim", "zscored", "zscored_prestim"]

# ── Polarity anchoring ────────────────────────────────────────
# Anchor each component's polarity to the ASSR electrodes before testing. A component's
# sign is fixed only globally by the decomposition's convention, which is NOT "positive
# means more fronto-central power" — the sign of a block's pattern weight over the mask
# electrodes splits across participants. Re-anchoring makes a higher value mean more 40 Hz
# power over that area for EVERY row, which is what lets the directional prior below be
# stated at all.
#
# Why this anchor is legitimate: it reads only the FORWARD PATTERN's projection onto a
# fixed electrode selection — a spatial property — never the tested response. An anchor
# taken from the tested quantity ("flip so Placebo is positive") breaks the exchangeability
# a paired test rests on and inflates the one-sided false-positive rate from 0.05 to ~0.68
# under a simulated null.
ALIGN_POLARITY_TO_MASK = True

# Direction of the 4b contrast, computed as Placebo - Psilocybin. The prior is that
# psilocybin LOWERS the 40 Hz response over the fronto-central area, i.e. a positive
# difference: "greater". One-sided halves the attainable p and forfeits any claim if the
# effect runs the other way; it is legitimate only because the direction was fixed in
# advance. The IC-vs-reference families stay two-sided (4c) or use their own prior (Step 9).
CONTRAST_ALTERNATIVE = "greater"

# ── Figures ───────────────────────────────────────────────────
SAVE_PLOTS = True
CONDITION_COLORS = {
    ConditionVariants.PLACEBO.value: "#0F6E8C",
    ConditionVariants.PSILOCYBIN.value: "#A6357F",
}
# Spread drawn around each mean time course, ACROSS PARTICIPANTS — never across trials:
# trials within a participant are correlated, so their spread understates the real
# uncertainty. "sem" is mean +/- standard error, "iqr" the 25-75 band around the median.
COURSE_SPREAD = "sem"
# Percentile bootstrap over participants for the forest intervals. The Wilcoxon p is exact
# and does NOT come from this; the interval is only there to show the spread.
N_BOOTSTRAP = 10_000
BOOTSTRAP_SEED = 42
ALPHA = 0.05

# ── Wavelet cache directories ─────────────────────────────────
WAVELET_DIR: Path = resolve_wavelet_dir(None, EXPERIMENT_NAME) / "broadband"
WAVELET_SUBSET_CACHE_DIR: Path = (
    resolve_notebook_wavelet_cache_dir(EXPERIMENT_NAME) / "broadband"
)
REUSE_WAVELET_SUBSET_CACHE = True

# ── Plots directory ───────────────────────────────────────────
JOIN_TAG = "joined_tracks" if JOIN is ConditionVariants.JOINED_TRACKS else "joined"
PLOTS_DIR: Path = (
    ProjectPaths.NOTEBOOKS_DIR
    / "07-ica-condition-comparison"
    / "plots"
    / EXPERIMENT_NAME.value
    / "broadband"
    / f"ica_component_analysis_{JOIN_TAG}"
    / f"ica_{N_ICA}"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

if TEST_SELECTION not in {name for name, _c, _h in FREQ_SELECTIONS}:
    raise ValueError(
        f"TEST_SELECTION {TEST_SELECTION!r} is not among FREQ_SELECTIONS "
        f"{[name for name, _c, _h in FREQ_SELECTIONS]}."
    )
for _variant in TEST_VARIANTS:
    if _variant not in SIGNAL_VARIANTS:
        raise ValueError(
            f"TEST_VARIANTS entries must be one of {SIGNAL_VARIANTS}; got {_variant!r}."
        )
for _variant, _source in VARIANT_REFERENCE_FROM.items():
    if _variant in TEST_VARIANTS and _source not in TEST_VARIANTS:
        raise ValueError(
            f"{_variant!r} takes its fixed-reference row from {_source!r}, so "
            f"{_source!r} has to be in TEST_VARIANTS too."
        )

print(f"Join            : {JOIN.value}  ({JOIN_TAG})")
print(f"Experiment      : {EXPERIMENT_NAME.value} / {MUSIC_TYPE.value}")
print(f"Conditions      : {[c.value for c in CONDITIONS_TO_POOL]}")
print(
    f"Extent          : channels={N_CHANNELS_SUBSET}, times={N_TIMES_SUBSET}, "
    f"pairs={N_PAIRS_SUBSET}"
)
print(f"Decomposition   : {N_ICA} ICs, {ICA_ALGORITHM}/{ICA_FUN}, seed {RANDOM_STATE}")
print(f"Selections      : {[name for name, _c, _h in FREQ_SELECTIONS]}")
print(
    f"Tests on        : {TEST_SELECTION}, response = {RESPONSE_MEASURE}, "
    f"variants {TEST_VARIANTS}"
)
print(f"Onsets from     : {CONCATENATED_DIR}")
print(f"Plots directory : {PLOTS_DIR}  (saving: {SAVE_PLOTS})")

## Data Loading — the raw, un-z-scored wavelet power of both conditions

Whichever join is selected, this cell produces the same three things: the **raw** wavelet
power per condition (participant-matched, row for row), the participant labels those rows
are in, and the channel names the filters will be indexed by.

The loader differs because the two joins have genuinely different requirements. The
recording-axis join stacks recordings into one subject axis, so it needs both conditions on
one time base (`load_joined_condition_wavelets`); the time-axis join lays them end to end
and does not (`load_paired_condition_wavelets`). The latter is asked for
`zscore_mode="none"`: it standardises each track before concatenating by default, and the
trials below must be cut from **raw** power — the standardisation is applied in Step 1,
where it belongs to the decomposition rather than to the signal.

**First run vs. every run after.** The source-of-truth wavelet tensor is decompressed in
full before it is trimmed to the requested extent, and the two conditions are loaded one at
a time, so the peak is one condition's cache. The trimmed result is written to
`WAVELET_SUBSET_CACHE_DIR` and later runs at the same extent read it back without opening
the big cache. Those entries are shared with every other wavelet workflow in this project.

In [ ]:
if JOIN is ConditionVariants.JOINED:
    pooled, condition_analyzers = load_joined_condition_wavelets(
        MUSIC_TYPE,
        EXCLUSION_CATEGORIES,
        FREQS,
        wavelet_dir=WAVELET_DIR,
        experiment_name=EXPERIMENT_NAME,
        representation=REPRESENTATION,
        conditions=CONDITIONS_TO_POOL,
        n_channels=N_CHANNELS_SUBSET,
        n_times=N_TIMES_SUBSET,
        reuse_wavelets=REUSE_WAVELETS,
        subset_cache_dir=WAVELET_SUBSET_CACHE_DIR,
        reuse_subset_cache=REUSE_WAVELET_SUBSET_CACHE,
    )
    if N_PAIRS_SUBSET is not None:
        keep = sorted(set(pooled.participants), key=participant_sort_key)[
            :N_PAIRS_SUBSET
        ]
        pooled = pooled.select_participants(keep)
    bb_ad = pooled.data
    container = pooled  # branch-independent handle on the loaded dataset
    # Row order WITHIN a condition block; condition_subjects returns both conditions'
    # rows in this same order, so the two are participant-matched row for row.
    participants = [
        p
        for p, c in zip(pooled.participants, pooled.subject_conditions)
        if c is pooled.conditions[0]
    ]
    raw_by_condition = {
        c.value: pooled.condition_subjects(bb_ad.data, c) for c in pooled.conditions
    }
    analyzer = condition_analyzers[pooled.conditions[0]]
else:
    paired, condition_analyzers = load_paired_condition_wavelets(
        MUSIC_TYPE,
        EXCLUSION_CATEGORIES,
        FREQS,
        wavelet_dir=WAVELET_DIR,
        experiment_name=EXPERIMENT_NAME,
        representation=REPRESENTATION,
        conditions=CONDITIONS_TO_POOL,
        # Raw power: the trials are cut from un-z-scored wavelet power, and the
        # decomposition's own standardisation is applied in Step 1.
        zscore_mode="none",
        n_channels=N_CHANNELS_SUBSET,
        n_times=N_TIMES_SUBSET,
        reuse_wavelets=REUSE_WAVELETS,
        subset_cache_dir=WAVELET_SUBSET_CACHE_DIR,
        reuse_subset_cache=REUSE_WAVELET_SUBSET_CACHE,
    )
    if N_PAIRS_SUBSET is not None and N_PAIRS_SUBSET < paired.n_pairs:
        keep = sorted(paired.participants, key=participant_sort_key)[:N_PAIRS_SUBSET]
        rows = [paired.participants.index(p) for p in keep]
        paired = dataclasses.replace(
            paired,
            data=dataclasses.replace(paired.data, data=paired.data.data[rows]),
            participants=tuple(keep),
        )
    bb_ad = paired.data
    container = paired
    participants = list(paired.participants)
    # Views into the concatenated tensor — deliberately not copies, which would double
    # the largest array in the notebook for nothing.
    raw_by_condition = {
        c.value: paired.condition_track(bb_ad.data, c) for c in paired.conditions
    }
    analyzer = condition_analyzers[paired.conditions[0]]

LABEL = bb_ad.label
sfreq = bb_ad.sfreq
n_participants = len(participants)
n_channels = raw_by_condition[CONDITIONS_TO_POOL[0].value].shape[1]
n_freqs = raw_by_condition[CONDITIONS_TO_POOL[0].value].shape[2]

if CAST_FLOAT32:
    # A no-op when the cache is already float32.
    raw_by_condition = {
        name: np.asarray(arr, dtype=np.float32)
        for name, arr in raw_by_condition.items()
    }

# ── The channel axis the filters and the electrode mask are indexed by ──
# Prefer the names the loader carried; fall back to the analyser's EEG picks in the same
# order the extent was trimmed to. Either way the length must match the data.
if bb_ad.feature_names is not None and len(bb_ad.feature_names) == n_channels:
    channel_names = list(bb_ad.feature_names)
else:
    channel_names = list(topo_info_subset(analyzer.info, n_channels)["ch_names"])
if len(channel_names) != n_channels:
    raise ValueError(
        f"{len(channel_names)} channel name(s) for a {n_channels}-channel axis; the "
        "electrode mask could not be aligned."
    )

if n_freqs != FREQS.size:
    raise ValueError(
        f"The loaded tensor has {n_freqs} frequency bin(s) but FREQS has {FREQS.size}; "
        "the cache was written on a different grid."
    )

print(f"Dataset      : {LABEL}   (join {JOIN.value})")
print(f"Participants : {n_participants}  {participants}")
print(
    f"Channels     : {n_channels}  (first={channel_names[0]}, last={channel_names[-1]})"
)
print(f"Frequencies  : {FREQS[0]:.1f}-{FREQS[-1]:.1f} Hz ({n_freqs} bins)")
print(f"Sampling     : {sfreq} Hz")
print("\nRaw wavelet power per condition (participants x channels x freqs x times):")
for c in CONDITIONS_TO_POOL:
    arr = raw_by_condition[c.value]
    print(
        f"  {c.value:<12}: {arr.shape}  {arr.dtype}  "
        f"{arr.nbytes / 1e9:.2f} GB  [{arr.min():.4g}, {arr.max():.4g}]"
    )
print("z-scoring    : none — this is the signal the trials are cut from")

## Dataset Selection — assemble the join, and the small helpers

`blocks` is the bookkeeping everything downstream indexes by: the identity of each channel
block on the stacked feature axis, in feature-axis order. That is the whole of the
difference between the two joins —

```
JOINED         blocks = [(participant, condition), ...]   B = 2P
JOINED_TRACKS  blocks = [(participant, None),      ...]   B = P
```

— and `block_of(participant, condition)` hides it, so a single line later reads "this
recording's filter" correctly under either join.

The standardisation is applied here, matching the decomposition notebooks exactly: each
`(block, channel, frequency)` series is z-scored along time. Under `JOINED_TRACKS` that
means z-scoring **each condition's track on its own and then concatenating**, which is the
`per_condition` mode — so an overall power difference between the conditions is normalised
away and what the components describe is temporal and spectral *structure*. The raw power
in `raw_by_condition` is untouched, and that is what the trials come from.

In [ ]:
# ── Assemble the join ─────────────────────────────────────────
# `assemble_join` does the standardisation AND the stacking, and carries the block
# bookkeeping that is the whole of the difference between the two joins:
#
#   JOINED         blocks = [(participant, condition), ...]   B = 2P
#   JOINED_TRACKS  blocks = [(participant, None),      ...]   B = P
#
# It is imported rather than written out here so this notebook and
# scripts/run_wavelet_jica.py decompose the identical matrix. Under JOINED_TRACKS it
# z-scores each condition's track on its own before concatenating (= "per_condition"),
# so an overall power difference between the conditions is normalised away and what the
# COMPONENTS describe is temporal and spectral structure. `raw_by_condition` is not
# touched, which is what keeps the raw power available for the trials.
layout = assemble_join(
    raw_by_condition,
    participants,
    [c.value for c in CONDITIONS_TO_POOL],
    JOIN,
)
blocks = layout.blocks
n_blocks = layout.n_blocks
n_features = n_blocks * n_channels
n_samples_ft = layout.n_freqs * layout.n_times
block_of = layout.block_of
ica_input = layout.matrix

print(
    f"Blocks        : {n_blocks}  "
    f"({'recording' if JOIN is ConditionVariants.JOINED else 'participant'} per block)"
)
print(f"FastICA input : {ica_input.shape}  (F*T samples, B*C mixing variables)")
print(f"                {ica_input.dtype}, {ica_input.nbytes / 1e9:.2f} GB")

# Every column must have unit variance — that is what makes the stacking fair, so it is
# checked rather than claimed.
_col_std = ica_input.std(axis=0)
print(
    f"Column std    : min {_col_std.min():.4f}, max {_col_std.max():.4f} "
    "(1.0 = no block can dominate the whitening by gain)"
)


# ── Small helpers used by several steps below ─────────────────
def _save(fig, name: str) -> None:
    """Write *fig* into PLOTS_DIR, when saving is on."""
    if not SAVE_PLOTS:
        return
    path = PLOTS_DIR / f"{name}.png"
    save_fig(fig, path)
    print(f"saved {path}")

---
## Step 1 — The joint ICA

The same fit the decomposition notebooks run, but called rather than written out, so this
notebook and [`scripts/run_wavelet_jica.py`](../../scripts/run_wavelet_jica.py) cannot
drift apart on what the components are.

**The channel reduction happens before FastICA, and it has to.** `FastICA` would whiten by
calling `scipy.linalg.svd` on the transposed input, and LAPACK indexes with **32-bit
integers** — a matrix with more than `2**31 - 1` elements is refused outright, which at
2340 features is ~918k samples (~73 s of both ASSR tracks on a 50-bin grid). So
[`fit_joint_ica`](../../src/analysis/wavelet_jica.py) takes the leading `N_ICA` directions
from the `(features, features)` **covariance** — 2340 × 2340 here, trivial — and runs
FastICA on the scores, composing the operators back into channel space afterwards. The two
routes span the *same* subspace (`u` and `d` of `svd(X.T)` are exactly the eigenvectors and
root-eigenvalues of `X.T X`; verified equal to 4e-16), so this is the same decomposition
reached by the arithmetic that fits the full recording.

Sign and order are pinned inside the fit — the largest absolute TF excursion positive
(hypothesis-free: it says nothing about 40 Hz), and components sorted by their share of the
joint channel-space energy — and the flip is applied to the sources, the patterns **and**
the filters together, so no downstream quantity can disagree about which way a component
points.

Read `converged` before anything else. A non-converged unmixing is wherever the solver
stopped rather than a fixed point, and no test below is worth running on one.

In [ ]:
result = fit_joint_ica(
    layout,
    n_ica=N_ICA,
    random_state=RANDOM_STATE,
    algorithm=ICA_ALGORITHM,
    fun=ICA_FUN,
    max_iter=ICA_MAX_ITER,
    tol=ICA_TOL,
)

# Unpacked into the names the steps below read. The two operators are NOT
# interchangeable:
#   `filters` is the BACKWARD model — what extracts a source from the channels, and
#     therefore the only thing that may be applied to a new signal.
#   `patterns` is the FORWARD model — how a source projects onto the channels, which is
#     what belongs on a topomap and what the polarity anchor reads. Using the wrong one
#     is the classic filter-vs-pattern error (Haufe et al., 2014, NeuroImage 87:96-110).
tf_maps = result.tf_maps  # (K, F, T of the join) — GLOBAL, one map per component
filters = result.filters  # (K, B, C)
patterns = result.patterns  # (B, K, C)
sources_ft = result.sources  # (F*T, K)
ic_variance = result.ic_variance  # (K,)
block_ic_energy = result.block_ic_energy  # (B, K) — the per-block loading
retained = result.retained
converged = result.converged
# Flat views of the same two operators, for the round-trip check in Step 2.
unmixing = filters.reshape(N_ICA, -1)  # (K, B*C)
mixing = patterns.transpose(1, 0, 2).reshape(N_ICA, -1).T  # (B*C, K)

print(f"Sources (F*T, K)  : {sources_ft.shape}")
print(f"TF maps (K, F, T) : {tf_maps.shape}  — one map per component, shared")
print(f"Filters (K, B, C) : {filters.shape}  — backward model, per block")
print(f"Patterns (B, K, C): {patterns.shape}  — forward model, per block")
print(
    f"Retained variance : {retained * 100:.1f}% of the joint channel space "
    f"({N_ICA} of {n_features} directions)"
)
print(
    f"FastICA converged : {converged} ({ICA_ALGORITHM}/{ICA_FUN}, "
    f"max_iter={ICA_MAX_ITER}, tol={ICA_TOL:g})"
)
if not converged:
    print(
        "  WARNING: the unmixing is wherever the solver stopped, not a fixed point. "
        "Lower N_ICA or try ICA_ALGORITHM='deflation' before reading anything below."
    )
print(
    f"Flipped {int((result.signs < 0).sum())}/{N_ICA} component(s); ordered by "
    "ic_variance."
)
print("\n  IC   ic_variance   loading spread over blocks (min-max % of IC)")
for k in range(N_ICA):
    share = block_ic_energy[:, k] / block_ic_energy[:, k].sum()
    print(
        f"  {k + 1:>3}   {ic_variance[k] * 100:9.2f}%   "
        f"{share.min() * 100:5.1f} - {share.max() * 100:5.1f}   "
        f"(even split {100 / n_blocks:.1f})"
    )

---
## Step 2 — Check the recovered filters, and be clear about what they are

Two things, and the second is the one to keep in mind while reading every figure below.

**The round trip.** `mixing_` is `pinv(components_)`, so `components_ @ mixing_` must be the
`K × K` identity. That is checked rather than assumed: a large residual would mean the
unmixing is rank-deficient and the per-block filters are not the operator the fit actually
used. Expect roughly `1e-7`–`1e-5`, not `1e-15` — the wavelet caches are float32, so the
pseudo-inverse is computed in that precision over `B·C` features, and an ill-conditioned
whitening inflates it further. (The IVA analysis notebooks have to *recover* the filter by
inverting the stored patterns; here both operators come straight out of the fit, so this is
a consistency check rather than a reconstruction.)

**What a per-block filter is.** `filters[k, b]` is row `k` of the unmixing matrix restricted
to block `b`'s channels, and

```
source_k(f, t) = Σ_b filters[k, b] · x_b(f, t)
```

so it is that block's **contribution** to the shared source, not its own copy of the
component. This matters in one specific way: a block whose loading on component `k` is
near zero contributes almost nothing to it, so the time course this notebook extracts for
that (participant, component) is near-noise — and it still enters the group mean and the
paired test with full weight. The loading column printed in Step 1 is what says which
(block, component) pairs those are; the per-participant panels of the decomposition
notebooks' bar plot are the same information drawn.

This is the price of jICA's one advantage over IVA on the sign side: because a component's
sign flips its map and its whole unmixing row together, there is no per-recording sign
ambiguity to resolve, and no risk that arbitrary flips manufacture a condition difference.

In [ ]:
# Expect a residual around 1e-7 to 1e-5, not 1e-15: the wavelet caches are float32, so
# sklearn computes mixing_ as a pseudo-inverse in that precision over B*C features, and
# an ill-conditioned whitening inflates it further. Orders of magnitude above that would
# mean the unmixing is genuinely rank-deficient.
IDENTITY_TOLERANCE = 1e-4
identity_error = float(np.abs(unmixing @ mixing - np.eye(N_ICA)).max())
print(
    f"|U A - I|  : {identity_error:.2e}  (K = {N_ICA}, tolerance "
    f"{IDENTITY_TOLERANCE:.0e})"
)
if identity_error > IDENTITY_TOLERANCE:
    raise ValueError(
        f"components_ @ mixing_ is not the identity (max residual {identity_error:.2e}); "
        "the unmixing is rank-deficient and the per-block filters are not the operator "
        "the fit used."
    )

# How much of each component's filter energy sits in each block — the same partition as
# the loading, read on the backward operator. A block with a near-zero row contributes
# almost nothing to that component, so its extracted time course is near-noise.
filter_share = (filters**2).sum(axis=2)  # (K, B)
filter_share = filter_share / filter_share.sum(axis=1, keepdims=True)
weakest = []
for k in range(N_ICA):
    b = int(np.argmin(filter_share[k]))
    weakest.append((k, blocks[b], filter_share[k, b]))

print("\nWeakest block per component (its share of that component's filter energy):")
for k, block, share in weakest:
    who = block[0] if block[1] is None else f"{block[0]} {block[1]}"
    print(f"  IC {k + 1:<3} {who:<22} {share:6.2%}   (even split {1 / n_blocks:.2%})")
_thin = [k + 1 for k in range(N_ICA) if filter_share[k].min() < 0.2 / n_blocks]
if _thin:
    print(
        f"\n  NOTE: IC {_thin} have at least one block contributing < 20% of an even "
        "share.\n  Those participants' extracted time courses are close to noise for "
        "that component,\n  and they still carry full weight in the group mean and the "
        "paired test."
    )

---
## Step 3 — The fixed ASSR-electrode reference

The include-list in
[`config/assr_electrodes/<coordinate system>.csv`](../../config/assr_electrodes) — the
standard fronto-central selection the 40 Hz steady-state response is read from — turned
into a `(channels,)` 0/1 vector by
[`assr_electrode_mask`](../../src/io/loading.py) and aligned to this notebook's channel
axis.

It is the same *kind* of operator as a learned filter: a weighting over channels that
contracts the channel axis and leaves frequency and time untouched. What makes it worth
having alongside them:

- **Fixed, not learned.** The weights come from the paradigm and the montage, so they are
  identical for every participant and every condition — nothing to estimate, nothing to
  sign-align, and no dependence on the decomposition having converged.
- **Non-negative.** It is an average over electrodes, so its power cannot go negative,
  which is why it is the one row a plain relative change is defined on (Step 6).
- **One row**, where the ICA gives `K`.

`ASSR_MASK_STRICT` decides what happens when a listed electrode is missing — preprocessing
drops the boundary electrodes, so this does happen. `True` refuses rather than quietly
averaging over fewer electrodes than the list names; relax it only after reading which
electrodes the log says are gone. The cell also refuses a selection so small that the
reference would be meaningless, which is what a leading channel subset produces.

In [ ]:
assr_mask = assr_electrode_mask(
    channel_names, COORDINATE_SYSTEM, strict=ASSR_MASK_STRICT
)
assr_channels = [name for name, keep in zip(channel_names, assr_mask) if keep]

# One filter row: the K = 1 analogue of `filters`, so the projection in Step 4 is the same
# contraction the learned filters get.
assr_filter = assr_mask.astype(float)
if ASSR_MASK_NORMALIZE:
    if assr_filter.sum() == 0:
        raise ValueError("The ASSR electrode selection is empty; nothing to average.")
    assr_filter = assr_filter / assr_filter.sum()

print(
    f"ASSR electrodes : {int(assr_mask.sum())} of {assr_mask.size} channel(s) "
    f"({'mean' if ASSR_MASK_NORMALIZE else 'sum'} over them)"
)
print(f"                  {assr_channels}")

# A handful of electrodes is not the fronto-central selection this reference is supposed
# to be — the situation a leading channel slice produces, where the mask happens to catch
# whatever sits at the start of the montage.
if int(assr_mask.sum()) < 5:
    raise ValueError(
        f"Only {int(assr_mask.sum())} ASSR electrode(s) are present on this "
        f"{n_channels}-channel axis, so the reference would not be the fronto-central "
        "selection it is meant to be. Set N_CHANNELS_SUBSET = None and re-run the "
        "loading cell: the selection is spread over the montage, so a leading channel "
        "slice does not contain it."
    )

# The full source axis: the learned components first, in component order, then the fixed
# reference last — so `sources[:, :N_ICA]` is still exactly the learned set.
SOURCE_LABELS = [f"IC {k + 1}" for k in range(N_ICA)] + [ASSR_MASK_LABEL]
n_sources = len(SOURCE_LABELS)
IC_LABELS = SOURCE_LABELS[:N_ICA]
MASK_INDEX = SOURCE_LABELS.index(ASSR_MASK_LABEL)
print(f"\nSources         : {n_sources}  {SOURCE_LABELS}")

---
## Step 4 — Project the tracks through both filters, at the 40 Hz band

The projection. For every participant and condition, that recording's block filter is
applied to that recording's own wavelet power:

```
source_k(f, t) = filters[k, b] @ x_b(:, f, t)      # (C,) @ (C,) -> scalar, per (f, t)
mask(f, t)     = assr_filter   @ x  (:, f, t)
```

Because a filter only contracts the channel axis, frequency and time pass through
untouched — which is why the **frequency collapse can be done first**, on the channel data,
and the result is identical: the band mean and the channel contraction act on different
axes and commute. That turns the projection into a small matrix product instead of one over
the whole `(C, F, T)` tensor, and is why this step is cheap.

The two frequency selections from `FREQ_SELECTIONS` are both extracted: the single 40 Hz
bin, and the mean over 35–45 Hz. The wide one is deliberately wider than the response — it
is insensitive to wavelet smearing and to a few Hz of stimulator drift, at the cost of
admitting some off-frequency power. Which one a result survives under is informative, so
both are carried through.

**Both signal variants are built here.** `prestim` projects the raw power (units of wavelet
power, normalised per trial in Step 6); `zscored` projects the power z-scored along time
first. The z-scoring is per `(recording, channel, frequency)`, so it commutes with slicing
the frequency axis but **not** with averaging over it — so for that variant the bins are
z-scored before the band mean, not after.

Everything lands on one source axis, `(participants, K + 1, times)`, which makes "the same
participant's same trial under each filter" a slice rather than a join.

In [ ]:
def _selection_bins(centre: float, halfwidth: float) -> np.ndarray:
    """Bin indices of ``centre +/- halfwidth`` on the wavelet frequency grid.

    A zero half-width is the single nearest bin. A positive one takes every bin in the
    closed interval, falling back to the nearest bin if the interval happens to fall
    between two — an empty selection would silently produce a NaN track.
    """
    if halfwidth < 0:
        raise ValueError(f"Half-width must be >= 0; got {halfwidth}.")
    if halfwidth == 0:
        return np.array([int(np.argmin(np.abs(FREQS - centre)))])
    inside = np.flatnonzero(
        (FREQS >= centre - halfwidth) & (FREQS <= centre + halfwidth)
    )
    return inside if inside.size else np.array([int(np.argmin(np.abs(FREQS - centre)))])


selection_bins = {
    name: _selection_bins(centre, halfwidth)
    for name, centre, halfwidth in FREQ_SELECTIONS
}
# Every bin any selection needs, so the frequency axis is cut ONCE. Slicing before
# z-scoring is exact: zscore_by_time standardises each frequency independently.
band_bins = np.array(sorted({int(b) for bins in selection_bins.values() for b in bins}))
local_bins = {
    name: np.array([int(np.flatnonzero(band_bins == b)[0]) for b in bins])
    for name, bins in selection_bins.items()
}

print(f"Frequency grid : {FREQS[0]:.1f}-{FREQS[-1]:.1f} Hz ({FREQS.size} bins)")
for name, _c, _h in FREQ_SELECTIONS:
    bins = selection_bins[name]
    print(f"  {name:<9}: bins {bins.tolist()} = {FREQS[bins].tolist()} Hz")
print(
    f"Bins kept      : {band_bins.tolist()} = "
    f"{FREQS[band_bins][0]:.0f}-{FREQS[band_bins][-1]:.0f} Hz"
)

# ── Cut the frequency axis once, per variant ──────────────────
# (P, C, n_band_bins, T) per condition. Small enough to keep both variants in memory.
band_data: dict[str, dict[str, np.ndarray]] = {}
for variant in TEST_VARIANTS:
    band_data[variant] = {}
    for c in CONDITIONS_TO_POOL:
        sliced = np.ascontiguousarray(raw_by_condition[c.value][:, :, band_bins, :])
        if VARIANT_RECIPE[variant]["zscore"]:
            # Per (recording, channel, frequency) along time — the decomposition's own
            # standardisation, and exact on the sliced bins. Applying the filter to
            # THIS is what makes the per-block term the component itself.
            sliced = zscore_by_time(sliced)
        band_data[variant][c.value] = sliced
    print(
        f"\n{variant:<9}: "
        + ", ".join(
            f"{c.value} {band_data[variant][c.value].shape} "
            f"{band_data[variant][c.value].nbytes / 1e9:.2f} GB"
            for c in CONDITIONS_TO_POOL
        )
    )


def project_sources(channel_band: np.ndarray, condition: str) -> np.ndarray:
    """Project one condition's band-collapsed channel data onto the source axis.

    :param channel_band: ``(P, C, T)`` band-collapsed wavelet power, rows in
        ``participants`` order.
    :param condition: Which condition these rows are, so the right block filter is used.
    :return: ``(P, n_sources, T)`` — the learned components, then the fixed reference.
    """
    n_times = channel_band.shape[-1]
    out = np.empty((n_participants, n_sources, n_times), dtype=float)
    for i, participant in enumerate(participants):
        b = block_of(participant, condition)
        # (K, C) @ (C, T) -> (K, T): contracts channels only.
        out[i, :N_ICA] = filters[:, b, :] @ channel_band[i]
        out[i, MASK_INDEX] = assr_filter @ channel_band[i]
    return out


# ── The band-collapsed source tracks, per variant and selection ──
source_tracks: dict[str, dict[str, dict[str, np.ndarray]]] = {}
for variant in TEST_VARIANTS:
    source_tracks[variant] = {}
    for name, _c, _h in FREQ_SELECTIONS:
        bins = local_bins[name]
        source_tracks[variant][name] = {
            c.value: project_sources(
                band_data[variant][c.value][:, :, bins, :].mean(axis=2), c.value
            )
            for c in CONDITIONS_TO_POOL
        }

print("\nSource tracks (participants x sources x times):")
for variant in TEST_VARIANTS:
    for name, _c, _h in FREQ_SELECTIONS:
        shapes = {
            c.value: source_tracks[variant][name][c.value].shape
            for c in CONDITIONS_TO_POOL
        }
        print(f"  {variant:<9} {name:<9}: {shapes}")

---
## Step 5 — Cut the tracks into stimulus-locked trials

The step that turns a continuous 40 Hz time course into the unit a trial-level analysis
works with: one fixed window per stimulus onset, **kept separately** rather than averaged.

**Where the onsets come from.** The sidecar of the concatenated array,
`data/processed/<experiment>/concatenated/<Condition>_ASSR.stimulus_onsets.npy`. That array
is the one the wavelet transform was run on, so its sample indices address exactly the time
axis these tracks are on — no shifting, no rescaling.

**The window comes from the paradigm, not from the recording.**
[`AssrEpoch`](../../src/definitions/constants.py) sets a `PRE_ONSET_S` baseline and a
`POST_ONSET_S` span (the stimulus plus an equally long tail), and the recording only *caps*
the latter: `iva_quality.onset_window` shortens `post` by the shortest observed inter-onset
gap so an epoch can never reach the next stimulus. Deriving the length from the gap alone
would make the window a property of whatever jitter this recording happened to have. The two
conditions then share the **shorter** `post`, so a Placebo trial and a Psilocybin trial are
the same number of samples and can be contrasted sample for sample.

**The cached extent is the binding constraint.** The full ASSR recording carries ~148 onsets
per condition; a 3000-sample (12 s) subset keeps only the onsets whose whole window fits
inside it — about nine. The cell prints how many that is against how many exist, because it
is the single most important caveat on everything downstream. Extending it means
re-projecting from a longer cache, not re-cutting these arrays.

**No baseline subtraction here.** The pre-onset samples are kept as they are, so the
correction stays a choice Step 6 makes rather than one already baked in.

In [ ]:
# ── The onsets, from the concatenated array's sidecar ─────────
sidecar_onsets = {}
for c in CONDITIONS_TO_POOL:
    onsets_path = CONCATENATED_DIR / (
        f"{c.value}_{MUSIC_TYPE.value}{ProjectPaths.STIMULUS_ONSETS_SUFFIX}"
    )
    if not onsets_path.exists():
        raise FileNotFoundError(
            f"No stimulus onsets for {c.value} at {onsets_path}. They are written next "
            "to the concatenated array by the stimulus alignment; without them there "
            "are no trials to cut."
        )
    sidecar_onsets[c.value] = np.load(onsets_path).astype(int)

# ── One epoch window, shared by both conditions ───────────────
geometry = {}
for c in CONDITIONS_TO_POOL:
    n_times_c = raw_by_condition[c.value].shape[-1]
    onsets = sidecar_onsets[c.value]
    inside = onsets[(onsets >= 0) & (onsets < n_times_c)]
    if inside.size == 0:
        raise ValueError(
            f"{c.value}: none of the {onsets.size} onset(s) falls inside the "
            f"{n_times_c}-sample cached track."
        )
    pre, post = iva_quality.onset_window(inside, n_times_c, sfreq)
    geometry[c.value] = (inside, pre, post, n_times_c)

_pres = {pre for _o, pre, _post, _n in geometry.values()}
if len(_pres) > 1:
    raise ValueError(
        f"Conditions disagree on the pre-onset baseline ({sorted(_pres)} samples); they "
        "cannot share an epoch window."
    )
EPOCH_PRE = _pres.pop()
EPOCH_POST = min(post for _o, _pre, post, _n in geometry.values())
EPOCH_LEN = EPOCH_PRE + EPOCH_POST
epoch_times = np.arange(-EPOCH_PRE, EPOCH_POST) / sfreq
stimulus_mask = AssrEpoch.stimulus_mask(epoch_times)
baseline_mask = epoch_times < 0.0


def _cut_trials(array: np.ndarray, onsets: np.ndarray):
    """One window per onset from the LAST axis, every trial kept.

    The per-trial counterpart of ``iva_quality.epoch_average``, which collapses the same
    windows to their mean. Windows overhanging either end are dropped, and the onsets that
    survived come back with the data so a trial index is never guessed.

    :return: ``(..., n_trials, EPOCH_LEN)`` — the trial axis sits just before time — and
        the kept onset samples.
    """
    n_times = array.shape[-1]
    kept = np.asarray(
        [
            int(o)
            for o in onsets
            if int(o) - EPOCH_PRE >= 0 and int(o) + EPOCH_POST <= n_times
        ],
        dtype=int,
    )
    if kept.size == 0:
        raise ValueError(
            f"No {EPOCH_LEN}-sample window fits inside the {n_times}-sample track."
        )
    return np.stack(
        [array[..., o - EPOCH_PRE : o + EPOCH_POST] for o in kept], axis=-2
    ), kept


trials: dict[str, dict[str, dict[str, np.ndarray]]] = {}
trial_onsets: dict[str, np.ndarray] = {}
for variant in TEST_VARIANTS:
    trials[variant] = {}
    for name, _c, _h in FREQ_SELECTIONS:
        trials[variant][name] = {}
        for c in CONDITIONS_TO_POOL:
            cut, kept = _cut_trials(
                source_tracks[variant][name][c.value], geometry[c.value][0]
            )
            trials[variant][name][c.value] = cut
            trial_onsets[c.value] = kept  # identical across variants and selections

print(
    f"Epoch window : {EPOCH_PRE} pre + {EPOCH_POST} post = {EPOCH_LEN} samples = "
    f"[{epoch_times[0]:.3f}, {epoch_times[-1]:.3f}] s @ {sfreq} Hz"
)
print(
    f"               stimulus 0-{AssrEpoch.STIMULUS_DURATION_S:.2f} s = "
    f"{int(stimulus_mask.sum())} sample(s); baseline "
    f"{int(baseline_mask.sum())} sample(s)"
)
print("\nOnsets available vs usable, per condition:")
for c in CONDITIONS_TO_POOL:
    inside, _pre, post, n_times_c = geometry[c.value]
    print(
        f"  {c.value:<12}: {sidecar_onsets[c.value].size} in the recording, "
        f"{inside.size} inside the cached {n_times_c} samples "
        f"({n_times_c / sfreq:.1f} s), {trial_onsets[c.value].size} whole window(s) kept"
    )
    if post < AssrEpoch.post_onset_samples(sfreq):
        print(
            f"                post trimmed to {post} samples ({post / sfreq:.3f} s) by "
            "the shortest inter-onset gap"
        )
if any(trial_onsets[c.value].size < MIN_TRIALS_EXPECTED for c in CONDITIONS_TO_POOL):
    print(
        f"\n  WARNING: fewer than MIN_TRIALS_EXPECTED ({MIN_TRIALS_EXPECTED}) trials in "
        "at least one condition. Raise N_TIMES_SUBSET."
    )

print("\nTrials (participants x sources x trials x samples):")
for variant in TEST_VARIANTS:
    for name, _c, _h in FREQ_SELECTIONS:
        shapes = {
            c.value: trials[variant][name][c.value].shape for c in CONDITIONS_TO_POOL
        }
        print(f"  {variant:<9} {name:<9}: {shapes}")

---
## Step 6 — Normalise the `prestim` trials, and check they look like a response

### The normalisation

Each `prestim` trial is referenced to its **own** pre-stimulus window rather than to a
condition- or participant-level average. That is what makes it worth doing: the level of
40 Hz power drifts across the recording and differs by participant, and a per-trial
baseline removes both without any group statistic entering the correction.

**Two forms, because one form does not fit both kinds of source.**

- **`z`** — `(x − baseline) / baseline SD`, each trial in units of its own pre-stimulus
  variability. Defined for **every** trial of **every** source regardless of sign,
  dimensionless, and comparable across participants and sources. **This is the canonical
  form, and the one every test runs on.**
- **`rel`** — `x / baseline − 1`, the relative change, as a readable percentage. Written
  **only where the baseline is positive**; every other trial is `NaN`, deliberately, so a
  downstream mean cannot quietly average a sign-flipped value.

The reason is structural, not cosmetic. The reference is a non-negative average over
electrodes, so dividing by its baseline is exactly the conventional relative-power change.
A learned filter is a **signed** combination of channels: its projected power runs negative
on a substantial share of trials, and some baselines sit near zero. Dividing by those does
not give a noisy answer, it gives a meaningless one — the sign flips wherever the baseline
is negative and the magnitude explodes wherever it is small. The cell tabulates how often
that happens per source, so the claim is measured rather than asserted.

Using one form for everything is the point: this analysis puts the learned filters and the
fixed reference side by side and asks which recovers the response better. Normalising them
differently would confound exactly that — a difference between the two arms could come from
the transform rather than from the filter, and the two arms would not even be in the same
units.

The `zscored` variant needs none of this: it was standardised along time before
projection, so it is already dimensionless and its pre-onset interval reads ≈ 0 by
construction. `zscored_prestim` *does* go through it — that is the whole point of the
variant — and its component rows are signed for the same reason the `prestim` ones are, so
`z` is the only usable form there too.

### The check

The last part of the cell is a **sanity check on the extraction, not the analysis**: the
group-and-trial mean of the reference row printed sample by sample. What is being checked is
that a stimulus-locked change exists at all and sits in the right place — rising after onset
and falling back around the 0.5 s stimulus offset. If it does not, the epochs are mis-timed
or the rows are mis-indexed, and no contrast below is worth running. No condition difference
is tested here.

Read the component rows of the QC table for *magnitude*, not sign: it is printed **before**
the polarity anchoring of Step 7, so a component that mirrors the reference shows up with
the opposite sign here and the right one in every figure below.

In [ ]:
# ── How often is a per-trial baseline unusable as a divisor? ──
print("Per-trial baselines, by source (a ratio needs a POSITIVE baseline):")
print(f"  {'source':<12} {'condition':<11} {'negative':>11} {'max |ratio|':>13}")
_probe = trials["prestim"][TEST_SELECTION] if "prestim" in TEST_VARIANTS else None
if _probe is not None:
    for s, source in enumerate(SOURCE_LABELS):
        for c in CONDITIONS_TO_POOL:
            arr = _probe[c.value][:, s]  # (P, N, W)
            base = arr[..., baseline_mask].mean(axis=-1)
            print(
                f"  {source:<12} {c.value:<11} "
                f"{f'{int((base < 0).sum())}/{base.size}':>11} "
                f"{np.abs(arr / base[..., None]).max():>13.3g}"
            )


def _normalise(array: np.ndarray):
    """Per-trial baseline normalisation of a ``(P, S, N, W)`` array.

    :return: ``(rel, z, baseline_positive)`` — the relative change (NaN where the
        baseline is not positive, because a negative baseline silently INVERTS the
        ratio), the baseline-SD z-score (defined for every trial whatever its sign), and
        the ``(P, S, N)`` mask of the trials ``rel`` is valid for.
    """
    base = array[..., baseline_mask].mean(axis=-1, keepdims=True)
    spread = array[..., baseline_mask].std(axis=-1, ddof=1, keepdims=True)
    positive = base > 0
    rel = np.where(positive, array / np.where(positive, base, 1.0) - 1.0, np.nan)
    usable = spread > 0
    z = np.where(usable, (array - base) / np.where(usable, spread, 1.0), np.nan)
    return rel, z, positive[..., 0]


# `signal` is what every later step reads: the canonical form of each variant.
signal: dict[str, dict[str, dict[str, np.ndarray]]] = {}
trials_rel: dict[str, dict[str, np.ndarray]] = {}
baseline_positive: dict[str, dict[str, np.ndarray]] = {}
for variant in TEST_VARIANTS:
    signal[variant] = {}
    for name, _c, _h in FREQ_SELECTIONS:
        signal[variant][name] = {}
        for c in CONDITIONS_TO_POOL:
            arr = trials[variant][name][c.value]
            if VARIANT_RECIPE[variant]["baseline"]:
                rel, z, positive = _normalise(arr)
                signal[variant][name][c.value] = z
                if name == TEST_SELECTION and variant == "prestim":
                    trials_rel[c.value] = rel
                    baseline_positive[c.value] = positive
            else:
                # Already dimensionless: z-scored along time before projection.
                signal[variant][name][c.value] = arr

# Hold the fixed-reference row still where a variant borrows it, so that variant is
# compared against exactly the same numbers as the one it borrows from.
for variant, source_variant in VARIANT_REFERENCE_FROM.items():
    if variant not in TEST_VARIANTS:
        continue
    for name, _c, _h in FREQ_SELECTIONS:
        for c in CONDITIONS_TO_POOL:
            signal[variant][name][c.value][:, MASK_INDEX] = signal[source_variant][
                name
            ][c.value][:, MASK_INDEX]
    print(
        f"{variant}: the {ASSR_MASK_LABEL} row is {source_variant}'s verbatim, so the "
        "two differ only in how the component was derived."
    )

print("\nShare of trials with a usable (positive) baseline, per source:")
for s, source in enumerate(SOURCE_LABELS):
    if not baseline_positive:
        break
    shares = {
        c.value: baseline_positive[c.value][:, s].mean() for c in CONDITIONS_TO_POOL
    }
    verdict = (
        "rel usable"
        if min(shares.values()) > 0.99
        else "USE z — rel is NaN on much of this row"
    )
    joined = "  ".join(f"{c} {v:5.0%}" for c, v in shares.items())
    print(f"  {source:<12} {joined}   {verdict}")

# ── QC: does the driven interval stand out from the baseline? ─
records = []
for variant in TEST_VARIANTS:
    for c in CONDITIONS_TO_POOL:
        z = signal[variant][TEST_SELECTION][c.value]  # (P, S, N, W)
        for s, source in enumerate(SOURCE_LABELS):
            records.append(
                {
                    "variant": variant,
                    "condition": c.value,
                    "source": source,
                    "stimulus": float(np.nanmean(z[:, s][..., stimulus_mask])),
                    "baseline": float(np.nanmean(z[:, s][..., baseline_mask])),
                }
            )
qc = pd.DataFrame.from_records(records)
print(
    f"\nStimulus- and baseline-interval means, {TEST_SELECTION}, "
    f"{n_participants} participants x ~{trial_onsets[CONDITIONS_TO_POOL[0].value].size} "
    "trials — QC of the extraction, not a test:"
)
display(
    qc.pivot_table(
        index=["variant", "source"],
        columns="condition",
        values=["baseline", "stimulus"],
    ).round(3)
)

# The shape over the epoch, on the reference row. A driven 40 Hz response should rise
# after onset and fall back around the stimulus offset; a monotonic drift across the whole
# window is a slow non-stationarity that per-trial baselining re-references to zero but
# does not remove.
marks = np.linspace(0, epoch_times.size - 1, 19).astype(int)
for variant in TEST_VARIANTS:
    print(
        f"\n[{variant}] {ASSR_MASK_LABEL} group+trial mean at "
        f"{FREQS[selection_bins[TEST_SELECTION]].tolist()} Hz:"
    )
    print("  t (s)  : " + " ".join(f"{epoch_times[i]:6.2f}" for i in marks))
    for c in CONDITIONS_TO_POOL:
        course = np.nanmean(
            signal[variant][TEST_SELECTION][c.value][:, MASK_INDEX], axis=(0, 1)
        )
        print(f"  {c.value[:6]:<6} : " + " ".join(f"{course[i]:6.2f}" for i in marks))
print(f"  stimulus interval: 0.00-{AssrEpoch.STIMULUS_DURATION_S:.2f} s")

---
## Step 7 — Participant-level tests

Two reductions, then the statistics. The signal arrives as
`(participants, sources, trials, samples)` and is collapsed twice:

```
(P, S, N, W)  --mean over the test window-->  (P, S, N)  --median over trials-->  (P, S)
```

and the **participants are the unit of every test below**. Not the trials: a participant's
trials are correlated, so a test that treats them as independent observations is
anticonservative. Aggregating first is not a loss of power either — for a balanced design
the paired test on participant summaries and a random-intercept mixed model give the same
standard error, because `Var = σ_b²/P + σ_w²/(P·m)` and the degrees of freedom are set by
the number of participants either way.

**Polarity is re-anchored to the ASSR electrodes first.** A component's sign is fixed only
by the decomposition's global convention, which is not "positive means more fronto-central
power" — the sign of a block's forward-pattern weight over the mask electrodes splits across
participants. So each block's component is flipped by

```
flip[b, k] = sign( mean forward-pattern weight of IC k over the ASSR electrodes )
```

after which a higher value means more 40 Hz power over that area for everyone, which is what
lets the directional prior be stated at all. This reads only the **pattern's** projection
onto a fixed electrode selection — a spatial property — never the tested response, so it
cannot manufacture a condition difference. Under `JOINED_TRACKS` the pattern is shared, so
the flip is identical for both conditions and the paired difference is untouched; under
`JOINED` it is resolved per recording from that recording's own pattern, which is still
independent of the tested quantity. The cell reports how well determined each anchor is
(`|w_mask| / |w_all|`): a component that barely projects onto those electrodes has an anchor
decided on noise.

**Exact Wilcoxon signed-rank.** The design is paired, so under the null a participant's
difference is equally likely to carry either sign; Wilcoxon enumerates all `2**P` sign
assignments over the **ranks** of `|d|`, making the *p*-value exact and free of any
distributional assumption. Ranking bounds how much any one participant can move the result,
which matters because the per-trial normalisation divides by a baseline SD estimated from a
handful of samples and is heavy-tailed across trials. The cost is that magnitude information
is discarded.

**Know the floor before reading the output.** The enumeration bounds how small a *p* can
get: `2/2**P` two-sided, half that one-sided, and only when every participant points the
same way. At `P = 12` that is 0.0005; at `P = 5` it is 0.06, i.e. the test cannot reject at
all. The cell prints the floor.

**Two families, both reported uncorrected.**

- **Condition contrast** — `Placebo − Psilocybin`, one test per source, one-sided
  `greater`: the prior is that psilocybin lowers the 40 Hz response over the fronto-central
  area. Meaningful only because the polarity anchor has made "higher = more power there"
  true of every row.
- **Component vs reference** — does a component separate the conditions better than the
  fixed electrode selection? The quantity is an **interaction**, not a within-condition
  comparison:

  ```
  d[p] = (placebo_ic − psilocybin_ic) − (placebo_mask − psilocybin_mask)
  ```

  Zero means the component separates Placebo from Psilocybin exactly as well as the mask
  does; positive means better. Two-sided, because nothing predicts that a learned component
  should beat a fixed selection. **Every component is tested, none selected** — picking one
  by its own effect size and then testing it on the same data would bias the result.

No multiplicity correction is applied, by choice: with a handful of tests per family, a *p*
just under 0.05 on one row is roughly what chance delivers. Read a row alongside its
`effect` and `same sign` columns and against its neighbours, not on its *p* alone.

**Why not test `|placebo − psilocybin|`.** It would remove the need for a direction and
would also destroy the test: `|d| ≥ 0` by construction, so "is it above zero?" is true
whenever there is any noise at all. The sign *is* the signal.

In [ ]:
# ── The test window ───────────────────────────────────────────
if TEST_STIMULUS_INTERVAL is None:
    test_stim_mask = stimulus_mask
    test_window_desc = f"0-{AssrEpoch.STIMULUS_DURATION_S:.3f} s (paradigm default)"
else:
    _lo, _hi = TEST_STIMULUS_INTERVAL
    test_stim_mask = (epoch_times >= _lo) & (epoch_times <= _hi)
    if not test_stim_mask.any():
        raise ValueError(
            f"TEST_STIMULUS_INTERVAL {TEST_STIMULUS_INTERVAL} selects no epoch sample; "
            f"the epoch spans [{epoch_times[0]:.3f}, {epoch_times[-1]:.3f}] s."
        )
    test_window_desc = f"{_lo:.3f}-{_hi:.3f} s (custom)"
test_rest_mask = ~test_stim_mask


def _reduce_window(windowed: np.ndarray) -> np.ndarray:
    """Collapse the epoch-time axis of a ``(..., W)`` array to one number per trial."""
    if RESPONSE_MEASURE == "stimulus":
        return windowed[..., test_stim_mask].mean(axis=-1)
    if RESPONSE_MEASURE == "stimulus_minus_rest":
        return windowed[..., test_stim_mask].mean(axis=-1) - windowed[
            ..., test_rest_mask
        ].mean(axis=-1)
    raise ValueError(
        f"RESPONSE_MEASURE must be 'stimulus' or 'stimulus_minus_rest'; got "
        f"{RESPONSE_MEASURE!r}."
    )


# ── One value and one epoch course per (participant, source) ──
value_by_variant: dict[str, dict[str, np.ndarray]] = {}
course_by_variant: dict[str, dict[str, np.ndarray]] = {}
for variant in TEST_VARIANTS:
    value_by_variant[variant] = {}
    course_by_variant[variant] = {}
    for c in CONDITIONS_TO_POOL:
        z = signal[variant][TEST_SELECTION][c.value]  # (P, S, N, W)
        value_by_variant[variant][c.value] = np.nanmedian(_reduce_window(z), axis=2)
        course_by_variant[variant][c.value] = np.nanmedian(z, axis=2)

# ── Re-anchor each block's component polarity to the ASSR area ──
# A property of the FORWARD PATTERN over a fixed electrode selection, so it never touches
# the tested response. Under JOINED_TRACKS the pattern is shared and the flip is identical
# for both conditions; under JOINED it is that recording's own.
pattern_weight = {}
pattern_scale = {}
flip_by_condition = {}
for c in CONDITIONS_TO_POOL:
    rows = [block_of(p, c.value) for p in participants]
    pattern_weight[c.value] = np.stack(
        [patterns[b][:, assr_mask].mean(axis=1) for b in rows]
    )  # (P, K)
    pattern_scale[c.value] = np.stack(
        [np.abs(patterns[b]).mean(axis=1) for b in rows]
    )  # (P, K)
    f = np.ones((n_participants, n_sources))
    if ALIGN_POLARITY_TO_MASK:
        # The reference row needs no flip: it is positively aligned with the ASSR area by
        # construction, so "higher = more power there" already holds.
        f[:, :N_ICA] = np.where(pattern_weight[c.value] >= 0, 1.0, -1.0)
    flip_by_condition[c.value] = f

value_by_variant = {
    var: {c: flip_by_condition[c] * arr for c, arr in vv.items()}
    for var, vv in value_by_variant.items()
}
course_by_variant = {
    var: {c: flip_by_condition[c][:, :, None] * arr for c, arr in cc.items()}
    for var, cc in course_by_variant.items()
}

print(
    f"Test frequency : {TEST_SELECTION}  "
    f"({FREQS[selection_bins[TEST_SELECTION]].tolist()} Hz)"
)
print(f"Test window    : {test_window_desc}, {int(test_stim_mask.sum())} sample(s)")
print(f"Response       : {RESPONSE_MEASURE}")
print("\nPolarity anchor — pattern weight over the ASSR electrodes:")
for c in CONDITIONS_TO_POOL:
    pw, ps, fl = (
        pattern_weight[c.value],
        pattern_scale[c.value],
        flip_by_condition[c.value],
    )
    print(f"  [{c.value}]  {'IC':<6} {'flipped':>9} {'med |w_mask|/|w_all|':>22}")
    for k in range(N_ICA):
        ratio = np.abs(pw[:, k]) / ps[:, k]
        strong = "ok" if np.median(ratio) > 0.2 else "WEAK"
        print(
            f"        IC {k + 1:<3} {int((fl[:, k] < 0).sum()):>6}/{n_participants} "
            f"{np.median(ratio):>22.3f}  {strong}"
        )
    if JOIN is ConditionVariants.JOINED_TRACKS:
        print("        (shared by both conditions — one pattern per participant)")
        break
if not ALIGN_POLARITY_TO_MASK:
    print(
        "  -> ALIGN_POLARITY_TO_MASK is False; the contrast direction is NOT "
        "interpretable."
    )


# ── Exact Wilcoxon signed-rank ────────────────────────────────
def paired_test(differences: np.ndarray, alternative: str) -> dict:
    """Exact Wilcoxon signed-rank test on per-participant differences.

    ``method="exact"`` refuses the normal approximation, so the p-value is exact. The unit
    is always the participant, never the trial.

    :param differences: One value per participant (non-finite entries are dropped).
    :param alternative: ``"two-sided"``, or a one-sided alternative where the direction
        was fixed in advance and the polarity anchoring makes it meaningful.
    :return: Median difference, robust effect size (median / its own MAD), how many
        participants point positive, the alternative used, and the p-value.
    """
    d = np.asarray(differences, dtype=float)
    d = d[np.isfinite(d)]
    n = d.size
    if n < 2:
        raise ValueError(f"Need at least 2 participants; got {n}.")
    result = wilcoxon(d, alternative=alternative, method="exact")
    spread = 1.4826 * np.median(np.abs(d - np.median(d)))
    return {
        "median": float(np.median(d)),
        "effect": float(np.median(d) / spread) if spread > 0 else np.nan,
        "same sign": f"{int((d > 0).sum())}/{n}",
        "alt": alternative,
        "p": float(result.pvalue),
    }


floor = 2.0 / 2**n_participants
print(
    f"\nTest          : exact Wilcoxon signed-rank over {n_participants} participants "
    f"(smallest attainable p = {floor:.5f} two-sided, {floor / 2:.5f} one-sided)"
)
if floor > 0.01:
    print(
        f"  WARNING: with {n_participants} participants no p can fall below "
        f"{floor:.4f}, so a null result here is a statement about the cohort size as "
        "much as about the effect. Set N_PAIRS_SUBSET = None."
    )

# Condition names as plain strings, the form the shared figures and the shared tests
# index by.
CONDITIONS_TO_POOL_VALUES = [c.value for c in CONDITIONS_TO_POOL]
c0 = CONDITIONS_TO_POOL[0].value
c1 = CONDITIONS_TO_POOL[1].value


def _run_tests(value: dict) -> tuple:
    """The condition contrast per source, and each IC's contrast against the reference."""
    rows = [
        {
            "source": source,
            **paired_test(value[c0][:, s] - value[c1][:, s], CONTRAST_ALTERNATIVE),
        }
        for s, source in enumerate(SOURCE_LABELS)
    ]
    contrast = pd.DataFrame(rows)

    mask_contrast = value[c0][:, MASK_INDEX] - value[c1][:, MASK_INDEX]
    rr = []
    for s, source in enumerate(IC_LABELS):
        ic_contrast = value[c0][:, s] - value[c1][:, s]
        rr.append(
            {
                "source": source,
                "IC contrast": float(np.nanmedian(ic_contrast)),
                **paired_test(ic_contrast - mask_contrast, "two-sided"),
            }
        )
    return contrast, pd.DataFrame(rr)


results_by_variant: dict[str, dict] = {}
for variant in TEST_VARIANTS:
    contrast, versus = _run_tests(value_by_variant[variant])
    results_by_variant[variant] = {"contrast": contrast, "versus": versus}

    print(f"\n{'=' * 74}")
    print(f"VARIANT: {variant}   ({TEST_SELECTION}, window {test_window_desc})")
    print("=" * 74)
    print(
        f"\nCondition contrast — {c0} minus {c1}, paired over {n_participants} "
        "participants:"
    )
    display(contrast.set_index("source").round({"median": 4, "effect": 3, "p": 5}))
    print(
        f"  Uncorrected p over {len(SOURCE_LABELS)} tests — read a row alongside "
        "`effect` and `same sign`, not on its p alone."
    )
    mc = (
        value_by_variant[variant][c0][:, MASK_INDEX]
        - value_by_variant[variant][c1][:, MASK_INDEX]
    )
    print(
        f"\nComponent vs {ASSR_MASK_LABEL} — (IC contrast) - (reference contrast); "
        f"0 = as good as the reference."
        f"\n     {ASSR_MASK_LABEL} own contrast: median {np.nanmedian(mc):+.4f}, "
        f"{int((mc > 0).sum())}/{n_participants} participants positive."
    )
    display(
        versus.set_index("source").round(
            {"IC contrast": 4, "median": 4, "effect": 3, "p": 5}
        )
    )
    better = int((versus["median"] > 0).sum())
    print(
        f"  Uncorrected p over {len(IC_LABELS)} tests; every IC tested. Direction: "
        f"{better}/{len(IC_LABELS)} IC(s) separate the conditions more than the "
        "reference."
    )

---
## Step 8 — The response over time, and the *p*-value summary

**Figure 1 — the 40 Hz response across stimuli, per spatial filter.** The figure this
notebook exists for. One panel per filter, both conditions overlaid. The line is the mean
across participants of their own **median-over-trials** time course; the band is the spread
**across participants**, never across trials — trials within a participant are correlated,
so a band drawn from them would look tight while saying nothing about how well the effect
generalises to a new person. The driven interval is shaded, and each panel carries its
one-sided *p* from Step 7 so the picture and the test are never read apart.

Under `prestim` the y axis **is** a signal-to-noise ratio: each trial's 40 Hz power in units
of its own pre-stimulus standard deviation, so `2` means "two pre-stimulus SDs above this
trial's own baseline" and `0` is the baseline itself. Under `zscored` it is the same
response in units of the whole track's variability instead.

The component panels are drawn **polarity-aligned**, the same flip the tests use: without
it a participant whose component loads negatively on the fronto-central electrodes would
cancel one who loads positively, and the group mean would collapse toward zero for reasons
that have nothing to do with the response. The reference panel is drawn in bold — it is the
assumption-free row, and the one to read first.

**Figure 2 — the *p*-value summary.**

- **Left: is Psilocybin lower than Placebo?** One row per source, showing the median paired
  difference `Placebo − Psilocybin` with a bootstrap interval, and every participant's own
  difference as a dot. Positive is the predicted direction, so the *p* is one-sided.
- **Right: does any component beat the reference?** The interaction from Step 7, two-sided,
  because nothing predicts which way it should go.

Intervals are a percentile bootstrap over participants and are there to convey spread — the
*p*-values come from the exact Wilcoxon test, not from the bootstrap, and the two can
disagree slightly at small `P`. Nothing is corrected for multiplicity, so read a marker
against its neighbours as much as against α.

Both figures are drawn once per entry of `TEST_VARIANTS` and saved with a `_<variant>`
suffix, so all three sit side by side. Read `prestim` against `zscored_prestim` first: they
are in the same units and judged against the same reference row, so any difference between
them is the cost of `prestim` approximating the component rather than reproducing it.

In [ ]:
# Both figures come from src/visualization/jica_plots.py rather than being drawn
# here, so this notebook and scripts/run_wavelet_jica.py produce the identical
# figure from the identical code. VARIANT_UNITS names each variant's y axis.
def draw_variant_figures(variant: str) -> None:
    """Figure 1 (epoch response) and Figure 2 (p-value summary) for one variant."""
    contrast_rows = results_by_variant[variant]["contrast"].to_dict("records")
    versus_rows = results_by_variant[variant]["versus"].to_dict("records")
    units = VARIANT_UNITS.get(variant, variant)
    note = (
        f"{n_participants} participants, ~{trial_onsets[c0].size} trials, "
        f"test window {test_window_desc} (polarity-aligned to the ASSR electrodes)"
    )

    fig = plot_response_courses(
        course_by_variant[variant],
        epoch_times,
        SOURCE_LABELS,
        CONDITIONS_TO_POOL_VALUES,
        contrast_rows,
        label=f"{LABEL}, variant {variant}",
        units=units,
        selection=TEST_SELECTION,
        condition_colors=CONDITION_COLORS,
        spread_mode=COURSE_SPREAD,
        test_interval=TEST_STIMULUS_INTERVAL,
        reference_label=ASSR_MASK_LABEL,
        alpha=ALPHA,
        note=note,
        save_path=(
            PLOTS_DIR / f"trial_course_by_source_{TEST_SELECTION}_{variant}.png"
            if SAVE_PLOTS
            else None
        ),
    )
    plt.show()
    plt.close(fig)

    fig = plot_pvalue_summary(
        value_by_variant[variant],
        SOURCE_LABELS,
        CONDITIONS_TO_POOL_VALUES,
        contrast_rows,
        versus_rows,
        label=f"{LABEL}, variant {variant}",
        units=units,
        reference_label=ASSR_MASK_LABEL,
        alpha=ALPHA,
        n_bootstrap=N_BOOTSTRAP,
        bootstrap_seed=BOOTSTRAP_SEED,
        note=f"floor p = {floor:.5f} — {note}",
        save_path=(
            PLOTS_DIR / f"pvalue_summary_{TEST_SELECTION}_{variant}.png"
            if SAVE_PLOTS
            else None
        ),
    )
    plt.show()
    plt.close(fig)


for variant in TEST_VARIANTS:
    print(f"\n{'#' * 26} FIGURES — variant: {variant} {'#' * 26}")
    draw_variant_figures(variant)

spread_label = (
    "mean +/- SEM across participants"
    if COURSE_SPREAD == "sem"
    else "median with 25-75 band across participants"
)
print(
    f"\nBands: {spread_label}. Figure-2 intervals are a {N_BOOTSTRAP:,}-draw "
    "percentile bootstrap over participants (spread only); the p-values are the exact "
    "Wilcoxon ones from Step 7."
)

---
## Step 9 — Is a component's SNR lower than the reference's, within each condition?

A **magnitude** comparison, and a different question from Step 7's. That one asked whether a
component separates the *conditions* better than the reference — an interaction. This asks
the plainer question the reference exists to answer: for the same stimulus-window response
the tests reduce to, is a learned component's SNR **lower** than the fixed electrode
average's? I.e. does the component recover *less* 40 Hz power over the fronto-central area
than simply averaging those electrodes does?

- **Quantity.** Per participant *and* condition, each source's response from Step 7 —
  baseline-referenced SNR in the `prestim` variant, the time-standardised response in
  `zscored`. Polarity-anchored to the ASSR electrodes, so a lower value is a genuine drop in
  recovered power, not a sign flip.
- **Test.** Paired exact Wilcoxon, one-sided `less`: `IC SNR − reference SNR < 0`, run
  **separately for Placebo and Psilocybin**, since the drug may change how well either filter
  recovers the response. Every (variant, condition, component) cell is tested, none selected.
- **Reading it.** One figure, one panel per variant; within a panel the two conditions are
  offset rows per component, coloured by condition, a **filled** marker meaning `p ≤ ALPHA`
  and a hollow one not. A marker left of zero means that component's SNR sits below the
  reference in that condition.

For jICA there is a structural reason to expect exactly that, and it is worth stating before
reading the panel: a per-block filter is one participant's **contribution** to a shared
source (Step 2), so it is not optimised to maximise that participant's own 40 Hz power the
way a fixed fronto-central average is aimed at it. A component sitting below the reference
here is therefore not automatically a failure of the decomposition — it is the expected cost
of the components being a group quantity. What would be informative is a component that
matches or beats the reference despite that.

In [ ]:
# Directional hypothesis: a learned component recovers LESS 40 Hz power over the ASSR area
# than the fixed electrode reference, i.e. (IC SNR - reference SNR) < 0. Tested separately
# per condition.
SNR_TEST_ALTERNATIVE = "less"

n_snr_tests = len(TEST_VARIANTS) * len(CONDITIONS_TO_POOL) * len(IC_LABELS)
snr_rows_by_variant = {}
print(
    f"SNR comparison — is a component's SNR {SNR_TEST_ALTERNATIVE} than "
    f"{ASSR_MASK_LABEL}'s?\n  {TEST_SELECTION} @ {test_window_desc}, paired over "
    f"{n_participants} participants, one-sided, per condition.\n"
)
for variant in TEST_VARIANTS:
    # reference_snr_tests is the same helper scripts/run_wavelet_jica.py calls, so the
    # notebook and the CLI cannot drift apart on what "below the reference" means.
    snr_rows_by_variant[variant] = {
        c.value: reference_snr_tests(
            value_by_variant[variant][c.value],
            SOURCE_LABELS,
            condition=c.value,
            alternative=SNR_TEST_ALTERNATIVE,
        )
        for c in CONDITIONS_TO_POOL
    }
    combined = pd.DataFrame(
        [row for rows in snr_rows_by_variant[variant].values() for row in rows]
    )
    n_sig = int((combined["p"] <= ALPHA).sum())
    print(f"[{variant}]  IC SNR vs {ASSR_MASK_LABEL}  (both conditions):")
    display(
        combined.set_index(["condition", "source"]).round(
            {"IC SNR": 4, "ref SNR": 4, "median": 4, "effect": 3, "p": 5}
        )
    )
    print(
        f"  {n_sig}/{len(combined)} (IC, condition) cell(s) significantly below the "
        f"reference (one-sided {SNR_TEST_ALTERNATIVE}, p <= {ALPHA}). Uncorrected over "
        f"{n_snr_tests} tests.\n"
    )

fig = plot_snr_vs_reference(
    value_by_variant,
    snr_rows_by_variant,
    SOURCE_LABELS,
    CONDITIONS_TO_POOL_VALUES,
    label=f"{LABEL}, {JOIN_TAG}",
    units_by_variant=VARIANT_UNITS,
    reference_label=ASSR_MASK_LABEL,
    condition_colors=CONDITION_COLORS,
    alternative=SNR_TEST_ALTERNATIVE,
    alpha=ALPHA,
    n_bootstrap=N_BOOTSTRAP,
    bootstrap_seed=BOOTSTRAP_SEED,
    note=f"n = {n_participants}, per condition, {TEST_SELECTION} @ {test_window_desc}",
    save_path=(
        PLOTS_DIR / f"snr_vs_reference_{TEST_SELECTION}_by_condition.png"
        if SAVE_PLOTS
        else None
    ),
)
plt.show()
plt.close(fig)